In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-qy4rptk4/unsloth_9c5f1a11fd2842a6a293eee19dd519d9
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-qy4rptk4/unsloth_9c5f1a11fd2842a6a293eee19dd519d9
  Resolved https://github.com/unslothai/unsloth.git to commit acc881452f11b7ec686c8a24d356fe014ef97551
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached xformers-0.0.26.post1.tar.gz (4.1 MB)
  Preparing metadata (setup.py) ... done
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for xformer

In [4]:
import torch
from unsloth import FastLanguageModel, is_bfloat16_supported

max_seq_length = 4096
dtype = None
load_in_4bit = True

print("Downloading Phi-4-mini...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "microsoft/phi-4-mini-instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Injecting LoRA Adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("Model ready!")

This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.


==((====))==  Unsloth 2026.3.11: Fast Phi3 patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

Injecting LoRA Adapters...
Unsloth: Making `model.base_model.model.model` require gradients
Model ready!


In [5]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
    mapping = {"role" : "role", "content" : "content", "user" : "user", "assistant" : "assistant"},
)

def formatting_prompts_func(examples):
    conversations = examples["conversations"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in conversations]
    return {"text": texts}

# Load the file you uploaded to the sidebar
dataset = load_dataset("json", data_files="jee_augmented_dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

# 5% Validation Split
split_dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print(f"Training on {len(train_dataset)} examples. Validating on {len(eval_dataset)} examples.")

Unsloth: Will map <|im_end|> to EOS = <|end|>.


Map:   0%|          | 0/850 [00:00<?, ? examples/s]

Training on 807 examples. Validating on 43 examples.


In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments

from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = eval_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 15,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        eval_strategy = "steps", # <--- THE FIX IS HERE
        eval_steps = 50,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("Starting Training...")
trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/807 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/43 [00:00<?, ? examples/s]

Starting Training...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 807 | Num Epochs = 2 | Total steps = 202
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,912,896 of 3,844,934,656 (0.23% trained)


Step,Training Loss,Validation Loss
50,0.586919,0.575713
100,0.549988,0.555681
150,0.543160,0.541776
200,0.483500,0.535204


Unsloth: Not an error, but Phi3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


In [7]:
import shutil

# Save the adapter locally in Colab
model.save_pretrained("phi-4-jee-math-lora")
tokenizer.save_pretrained("phi-4-jee-math-lora")

# Zip the folder so you can download it easily
shutil.make_archive("phi-4-jee-math-lora", 'zip', "phi-4-jee-math-lora")
print("Saved and zipped! Look in the left sidebar for 'phi-4-jee-math-lora.zip'.")
print("Right-click it and press 'Download' to save it to your laptop.")

Saved and zipped! Look in the left sidebar for 'phi-4-jee-math-lora.zip'.
Right-click it and press 'Download' to save it to your laptop.


In [12]:
from unsloth.chat_templates import get_chat_template
from transformers import TextStreamer

# 1. Flip Unsloth into 2x Faster Inference Mode
FastLanguageModel.for_inference(model)

test_question = "Find the shortest distance between the parallel lines 3x - 4y + 7 = 0 and 3x - 4y - 5 = 0."

messages = [
    {"role": "user", "content": test_question}
]

# 2. THE FIX: Separate formatting and tokenization
# First, render the raw text with the <|im_start|> tags
prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize = False, # Keep it as a string first
    add_generation_prompt = True,
)

# Second, tokenize it explicitly into a PyTorch dictionary
inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")

print("\n" + "="*50)
print(f"USER QUESTION: {test_question}")
print("="*50 + "\n")
print("MODEL OUTPUT:\n")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

# 3. Unpack the inputs dictionary using **inputs
_ = model.generate(
    **inputs, # <--- This safely unpacks the input_ids and attention_mask tensors
    streamer = text_streamer,
    max_new_tokens = 2000,
    temperature = 0.1,
    use_cache = True
)

Both `max_new_tokens` (=2000) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



USER QUESTION: Find the shortest distance between the parallel lines 3x - 4y + 7 = 0 and 3x - 4y - 5 = 0.

MODEL OUTPUT:

<think>
The shortest distance between two parallel lines can be found using the formula: d = |c2 - c1| / √(a^2 + b^2), where a and b are the coefficients of x and y respectively, and c1 and c2 are the constants in the equations of the lines. For the given lines, a = 3, b = -4, c1 = 7, and c2 = -5. Substituting these values into the formula gives: d = |-5 - 7| / √(3^2 + (-4)^2) = 12 / √(9 + 16) = 12 / √25 = 12 / 5 = 2.4
</think>
\boxed{2.4}<|im_end|>


In [11]:
from google.colab import drive
import shutil

# 1. Mount your Google Drive
drive.mount('/content/drive')

# 2. Copy the zip file directly to the root of your Google Drive
source_file = "phi-4-jee-math-lora.zip"
destination = "/content/drive/MyDrive/phi-4-jee-math-lora.zip"

try:
    shutil.copy(source_file, destination)
    print("-> SUCCESS! The file is now safely in your Google Drive.")
except FileNotFoundError:
    print("-> ERROR: The zip file wasn't found. Did you run the shutil.make_archive cell?")

Mounted at /content/drive
-> SUCCESS! The file is now safely in your Google Drive.
